"""
# Director and Cast Consistency Analysis
## Performance Benchmarking Implementation

### Analysis Overview:
- **Q1. Director–Genre Frequency**: High-cardinality GroupBy
- **Q2. Director Success Rate by Genre**: Multi-stage aggregation
- **Q3. Cast Genre Specialization**: Pattern detection with Filter + GroupBy
- **Q4. Director Versatility Score**: Distinct-count operation
- **Q5. Recurring Director–Cast Combinations**: Multi-way join complexity
"""

In [2]:
import pandas as pd
import numpy as np
import time
import psutil
import os
import ast
from datetime import datetime


In [3]:
print("\nSTARTING ANALYSIS")
print("-"*40)

start_total = time.time()
start_memory_total = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu_total = psutil.Process(os.getpid()).cpu_percent(interval=None)


STARTING ANALYSIS
----------------------------------------


In [4]:
# Load Data with Benchmarking
print("\nLOADING DATA")
print("-"*40)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

movies = pd.read_csv(
    "movies_cleaned.csv",
    dtype={
        "vote_average": "float32",
        "vote_count": "int32"
    }
)

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory

# Store metrics for summary
load_time = execution_time
load_memory = memory_delta
load_end_memory = end_memory

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"Rows loaded: {len(movies):,}")
print(f"Columns: {len(movies.columns)}")
print(f"Initial Memory (MB): {round(movies.memory_usage(deep=True).sum()/1024**2, 2):,}")


LOADING DATA
----------------------------------------

 BENCHMARKS:
Time: 9.42 seconds
Memory delta: 2651.52 MB
Memory total: 2784.33 MB
Rows loaded: 590,202
Columns: 166
Initial Memory (MB): 2,131.44


In [5]:
# DATA PREPARATION
print("\nPREPARING DATA")
print("-"*40)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["year"] = movies["release_date"].dt.year.astype("int16")

def parse_list(x):
    if isinstance(x, str):
        try:
            if x.startswith("["):
                return ast.literal_eval(x)
            return [i.strip() for i in x.split(",")]
        except:
            return ["Unknown"]
    return x

movies["genres"] = movies["genres"].apply(parse_list)
movies["cast"] = movies["cast"].apply(parse_list)
movies["director"] = movies["director"].apply(parse_list)

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")


PREPARING DATA
----------------------------------------

 BENCHMARKS:
Time: 2.17 seconds
Memory delta: 639.69 MB
Memory total: 3424.20 MB


In [6]:
print("\n" + "="*80)
print("Q1: DIRECTOR–GENRE FREQUENCY")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Explode director and genres
dg = movies.explode("director").explode("genres")

# Group by to get frequency
director_genre_freq = (
    dg.groupby(["director", "genres"])
      .size()
      .reset_index(name="count")
)

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(director_genre_freq)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis1_time = execution_time
analysis1_memory = memory_delta

print(f"\n RESULTS:")
print(f"Total director-genre pairs: {rows_processed:,}")
print(f"Unique directors: {director_genre_freq['director'].nunique():,}")
print(f"Unique genres: {director_genre_freq['genres'].nunique()}")
print(director_genre_freq.head(10))

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Groups created: {rows_processed:,}")
print(f"Groups/second: {rows_per_second:,.0f}")


Q1: DIRECTOR–GENRE FREQUENCY

 RESULTS:
Total director-genre pairs: 599,123
Unique directors: 254,882
Unique genres: 19
                director       genres  count
0                               Crime      1
1                         Documentary      2
2                               Drama      1
3                               Music      1
4                            Thriller      1
5          "Slimer" Joel  Documentary      1
6  "Timebomb" Tom Hilton       Comedy      1
7  "Timebomb" Tom Hilton  Documentary      1
8  "Timebomb" Tom Hilton        Drama      1
9  "Timebomb" Tom Hilton       Family      1

 BENCHMARKS:
Time: 3.36 seconds
Memory delta: 1002.50 MB
Memory total: 4426.83 MB
CPU: 0.0%
Groups created: 599,123
Groups/second: 178,350


In [7]:
print("\n" + "="*80)
print("Q2: DIRECTOR SUCCESS RATE BY GENRE")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Multi-stage aggregation
director_success = (
    dg.groupby(["director", "genres"])
      .agg(
          avg_rating=("vote_average", "mean"),
          total_movies=("id", "count"),
          max_rating=("vote_average", "max"),
          min_rating=("vote_average", "min")
      )
      .round(2)
      .reset_index()
)

# Filter for directors with at least 2 movies in a genre
director_success = director_success[director_success["total_movies"] >= 2]

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(director_success)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis2_time = execution_time
analysis2_memory = memory_delta

print(f"\n RESULTS:")
print(f"Total director-genre success records: {rows_processed:,}")
print(f"Unique directors with success metrics: {director_success['director'].nunique():,}")
print(f"Unique genres covered: {director_success['genres'].nunique()}")
print(director_success.head(10))

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Groups created: {rows_processed:,}")
print(f"Groups/second: {rows_per_second:,.0f}")


Q2: DIRECTOR SUCCESS RATE BY GENRE

 RESULTS:
Total director-genre success records: 162,293
Unique directors with success metrics: 80,774
Unique genres covered: 19
                    director       genres  avg_rating  total_movies  \
1                             Documentary         3.9             2   
23       'Weird Al' Yankovic        Music         4.0             4   
24  'Xiongzaixia' Tan Jiahao  Documentary         2.6             5   
35                    2chain       Comedy         0.0             2   
37                    2chain  Documentary         0.0             2   
54              @opium_latte  Documentary         0.0            10   
59               A Abdulkani        Drama         0.0             3   
61               A Abdulkani       Horror         0.0             3   
62               A Abdulkani      Mystery         0.0             3   
71                      A Da    Animation         7.7             2   

    max_rating  min_rating  
1          7.8        0.

In [8]:
print("\n" + "="*80)
print("Q3: CAST GENRE SPECIALIZATION")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Explode cast and genres (this is memory intensive)
cg = movies.explode("cast").explode("genres")

# Group by to find cast genre frequencies
cast_genre = (
    cg.groupby(["cast", "genres"])
      .size()
      .reset_index(name="count")
)

# Filter for specialized cast (more than 5 movies in same genre)
specialized_cast = cast_genre[cast_genre["count"] > 5].sort_values("count", ascending=False)

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(cast_genre)
specialized_count = len(specialized_cast)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis3_time = execution_time
analysis3_memory = memory_delta

print(f"\n RESULTS:")
print(f"Total cast-genre pairs: {rows_processed:,}")
print(f"Specialized cast-genre pairs (>5 movies): {specialized_count:,}")
print(f"Unique specialized cast members: {specialized_cast['cast'].nunique():,}")
print(f"Unique specialized genres: {specialized_cast['genres'].nunique()}")
print("\nTop 10 specialized cast-genre combinations:")
print(specialized_cast.head(10))

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Total cast-genre pairs: {rows_processed:,}")
print(f"Pairs/second: {rows_per_second:,.0f}")


Q3: CAST GENRE SPECIALIZATION

 RESULTS:
Total cast-genre pairs: 5,381,320
Specialized cast-genre pairs (>5 movies): 324,686
Unique specialized cast members: 132,223
Unique specialized genres: 19

Top 10 specialized cast-genre combinations:
                      cast     genres  count
3423299          Mel Blanc  Animation   1047
3423300          Mel Blanc     Comedy    805
2521337                Jr.      Drama    424
1611552       Frank Welker  Animation    374
577457        Bess Flowers     Comedy    373
2103776        Jack Mercer  Animation    360
2129271  Jagathy Sreekumar      Drama    359
1611557       Frank Welker     Family    335
577460        Bess Flowers      Drama    319
4733278           Sukumari      Drama    312

 BENCHMARKS:
Time: 42.85 seconds
Memory delta: -1284.61 MB
Memory total: 3142.78 MB
CPU: 0.0%
Total cast-genre pairs: 5,381,320
Pairs/second: 125,581


In [9]:
print("\n" + "="*80)
print("Q4: DIRECTOR VERSATILITY SCORE")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Calculate distinct genres per director
director_versatility = (
    dg.groupby("director")["genres"]
      .nunique()
      .reset_index(name="versatility_score")
)

# Add movie count for context
director_movie_count = (
    dg.groupby("director")["id"]
      .count()
      .reset_index(name="total_movies")
)

# Merge the dataframes
director_versatility = director_versatility.merge(director_movie_count, on="director")

# Calculate average movies per genre
director_versatility["avg_movies_per_genre"] = (
    director_versatility["total_movies"] / director_versatility["versatility_score"]
).round(2)

# Filter for directors with at least 3 movies
director_versatility = director_versatility[director_versatility["total_movies"] >= 3]

# Sort by versatility score (most versatile first)
director_versatility = director_versatility.sort_values("versatility_score", ascending=False)

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(director_versatility)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis4_time = execution_time
analysis4_memory = memory_delta

print(f"\n RESULTS:")
print(f"Total directors with versatility scores: {rows_processed:,}")
print(f"Average versatility score: {director_versatility['versatility_score'].mean():.2f}")
print(f"Max versatility score: {director_versatility['versatility_score'].max()}")
print(f"Min versatility score: {director_versatility['versatility_score'].min()}")
print("\nTop 10 most versatile directors:")
print(director_versatility.head(10))

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Directors analyzed: {rows_processed:,}")
print(f"Directors/second: {rows_per_second:,.0f}")


Q4: DIRECTOR VERSATILITY SCORE

 RESULTS:
Total directors with versatility scores: 100,162
Average versatility score: 3.94
Max versatility score: 19
Min versatility score: 1

Top 10 most versatile directors:
                    director  versatility_score  total_movies  \
201521       Robert Zemeckis                 19            81   
201387      Robert Stevenson                 19           133   
120123                   Jr.                 19           265   
106349        Jean Yarbrough                 18           157   
59763          Dmitri Frolov                 18           110   
163514  Michael Lindsay-Hogg                 18            85   
244546      William Beaudine                 18           380   
21855           Arthur Lubin                 18           150   
44492            Chuck Jones                 17           630   
201494           Robert Wise                 17            92   

        avg_movies_per_genre  
201521                  4.26  
201387       

In [10]:
print("\n" + "="*80)
print("Q5: RECURRING DIRECTOR–CAST COMBINATIONS")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Explode director and cast only (no genres)
dc = movies.explode("director").explode("cast")

# Group to find collaboration frequency
director_cast_pairs = (
    dc.groupby(["director", "cast"])
      .size()
      .reset_index(name="collaboration_count")
)

# Filter for recurring pairs (more than 2 collaborations)
recurring_pairs = director_cast_pairs[
    director_cast_pairs["collaboration_count"] > 2
].sort_values("collaboration_count", ascending=False)

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(director_cast_pairs)
recurring_count = len(recurring_pairs)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis5_time = execution_time
analysis5_memory = memory_delta

print(f"\n RESULTS:")
print(f"Total director-cast pairs: {rows_processed:,}")
print(f"Recurring pairs (>2 collaborations): {recurring_count:,}")
print(f"Unique directors in recurring pairs: {recurring_pairs['director'].nunique():,}")
print(f"Unique cast members in recurring pairs: {recurring_pairs['cast'].nunique():,}")
print(f"Maximum collaborations: {recurring_pairs['collaboration_count'].max()}")
print("\nTop 10 most frequent director-cast collaborations:")
print(recurring_pairs.head(10))

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Total pairs analyzed: {rows_processed:,}")
print(f"Pairs/second: {rows_per_second:,.0f}")


Q5: RECURRING DIRECTOR–CAST COMBINATIONS

 RESULTS:
Total director-cast pairs: 5,928,558
Recurring pairs (>2 collaborations): 181,112
Unique directors in recurring pairs: 24,827
Unique cast members in recurring pairs: 89,128
Maximum collaborations: 320

Top 10 most frequent director-cast collaborations:
                    director                  cast  collaboration_count
1854988  Gilbert M. Anderson   Gilbert M. Anderson                  320
1723635         Friz Freleng             Mel Blanc                  254
966028           Chuck Jones             Mel Blanc                  235
3024788           Kevin Dunn             John Cena                  191
4677193      Robert McKimson             Mel Blanc                  179
3025107           Kevin Dunn           Randy Orton                  174
3024683           Kevin Dunn          Glenn Jacobs                  167
3024938           Kevin Dunn          Mark Calaway                  165
3025086           Kevin Dunn            Paul W

In [11]:
print("\n" + "="*80)
print("PERFORMANCE SUMMARY")
print("="*80)

# Collect results from all analyses
results = [
    {"Analysis": "Data Loading", "Time (s)": load_time if 'load_time' in locals() else 0, 
     "Memory (MB)": load_memory if 'load_memory' in locals() else 0},
    
    {"Analysis": "Data Preparation", "Time (s)": globals().get('prep_time', 0), 
     "Memory (MB)": globals().get('prep_memory', 0)},
    
    {"Analysis": "Q1: Director-Genre Frequency", "Time (s)": analysis1_time if 'analysis1_time' in locals() else 0, 
     "Memory (MB)": analysis1_memory if 'analysis1_memory' in locals() else 0},
    
    {"Analysis": "Q2: Director Success Rate", "Time (s)": analysis2_time if 'analysis2_time' in locals() else 0, 
     "Memory (MB)": analysis2_memory if 'analysis2_memory' in locals() else 0},
    
    {"Analysis": "Q3: Cast Genre Specialization", "Time (s)": analysis3_time if 'analysis3_time' in locals() else 0, 
     "Memory (MB)": analysis3_memory if 'analysis3_memory' in locals() else 0},
    
    {"Analysis": "Q4: Director Versatility", "Time (s)": analysis4_time if 'analysis4_time' in locals() else 0, 
     "Memory (MB)": analysis4_memory if 'analysis4_memory' in locals() else 0},
    
    {"Analysis": "Q5: Director-Cast Combinations", "Time (s)": analysis5_time if 'analysis5_time' in locals() else 0, 
     "Memory (MB)": analysis5_memory if 'analysis5_memory' in locals() else 0}
]

# Create DataFrame for better display
summary_df = pd.DataFrame(results)

# Calculate totals
total_time = summary_df["Time (s)"].sum()
total_memory = summary_df["Memory (MB)"].sum()

print("\n" + "-"*60)
print(f"{'Analysis':<40} {'Time (s)':<12} {'Memory (MB)':<12}")
print("-"*60)

for _, row in summary_df.iterrows():
    print(f"{row['Analysis']:<40} {row['Time (s)']:<12.2f} {row['Memory (MB)']:<12.2f}")

print("-"*60)
print(f"{'TOTAL':<40} {total_time:<12.2f} {total_memory:<12.2f}")
print("-"*60)

# Calculate end time and memory
end_total = time.time()
end_memory_total = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu_total = psutil.Process(os.getpid()).cpu_percent(interval=None)

total_execution_time = end_total - start_total
total_memory_delta = end_memory_total - start_memory_total

# Additional insights
print("\n" + "="*60)
print("INSIGHTS")
print("="*60)
if total_time > 0:
    max_time_idx = summary_df['Time (s)'].idxmax()
    max_memory_idx = summary_df['Memory (MB)'].idxmax()
    print(f"• Most time-consuming: {summary_df.loc[max_time_idx, 'Analysis']} ({summary_df['Time (s)'].max():.2f}s)")
    print(f"• Most memory-intensive: {summary_df.loc[max_memory_idx, 'Analysis']} ({summary_df['Memory (MB)'].max():.2f}MB)")
    print(f"• Average time per analysis: {total_time/len(results):.2f}s")
print(f"• Total dataset size: {len(movies):,} rows" if 'movies' in locals() else "• Total dataset size: N/A")
print(f"• Total execution time: {total_execution_time:.2f} seconds")
print(f"• Peak memory usage: {end_memory_total:.2f} MB")
print(f"• Total memory delta: {total_memory_delta:.2f} MB")
print("="*80)


PERFORMANCE SUMMARY

------------------------------------------------------------
Analysis                                 Time (s)     Memory (MB) 
------------------------------------------------------------
Data Loading                             9.42         2651.52     
Data Preparation                         0.00         0.00        
Q1: Director-Genre Frequency             3.36         1002.50     
Q2: Director Success Rate                0.38         0.39        
Q3: Cast Genre Specialization            42.85        -1284.61    
Q4: Director Versatility                 0.53         -70.23      
Q5: Director-Cast Combinations           15.74        1321.47     
------------------------------------------------------------
TOTAL                                    72.29        3621.03     
------------------------------------------------------------

INSIGHTS
• Most time-consuming: Q3: Cast Genre Specialization (42.85s)
• Most memory-intensive: Data Loading (2651.52MB)
• Average

### Mapreduce Implementation for qeuries 1 , 3 and 5 

In [13]:
from pyspark.sql import SparkSession

try:
    spark
except NameError:
    spark = SparkSession.builder \
        .appName("MapReduceAnalysis") \
        .master("local[*]") \
        .getOrCreate()

sc = spark.sparkContext

print("Spark ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 11:51:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/19 11:51:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark ready


In [14]:
# ============================================================
# MAPREDUCE - Q1: DIRECTOR-GENRE FREQUENCY
# ============================================================

start_time = time.time()

records = movies[["director", "genres"]].to_dict("records")
rdd = sc.parallelize(records)

def emit_pairs(row):
    return [((d, g), 1) for d in row["director"] for g in row["genres"]]

result_q1 = (
    rdd.flatMap(emit_pairs)
       .reduceByKey(lambda a, b: a + b)
       .map(lambda x: (x[0][0], x[0][1], x[1]))
)

df_q1_mr = pd.DataFrame(result_q1.collect(), columns=["director", "genre", "count"]) \
            .sort_values("count", ascending=False)

end_time = time.time()

print("Top results:")
print(df_q1_mr.head(10))
print(f"\nTime: {end_time - start_time:.2f}s")

26/03/19 11:51:37 WARN TaskSetManager: Stage 0 contains a task of very large size (2844 KiB). The maximum recommended task size is 1000 KiB.


Top results:
                   director      genre  count
242300       Dave Fleischer  Animation    364
302590           Kevin Dunn     Action    340
422139           Kevin Dunn      Drama    309
368236  Gilbert M. Anderson    Western    305
301684         Friz Freleng  Animation    293
480160          Chuck Jones  Animation    276
419032        D.W. Griffith      Drama    259
182780          Jules White     Comedy    251
182393       Dave Fleischer     Comedy    250
121721         Friz Freleng     Comedy    234

Time: 3.13s


In [15]:
# ============================================================
# MAPREDUCE - Q3: CAST GENRE SPECIALIZATION
# ============================================================

start_time = time.time()

records = movies[["cast", "genres"]].to_dict("records")
rdd = sc.parallelize(records)

def emit_pairs(row):
    return [((c, g), 1) for c in row["cast"] for g in row["genres"]]

result_q3 = (
    rdd.flatMap(emit_pairs)
       .reduceByKey(lambda a, b: a + b)
       .filter(lambda x: x[1] > 5)
       .map(lambda x: (x[0][0], x[0][1], x[1]))
)

df_q3_mr = pd.DataFrame(result_q3.collect(), columns=["cast", "genre", "count"]) \
            .sort_values("count", ascending=False)

end_time = time.time()

print("Top specialized cast:")
print(df_q3_mr.head(10))
print(f"\nTime: {end_time - start_time:.2f}s")

26/03/19 11:51:40 WARN TaskSetManager: Stage 2 contains a task of very large size (21430 KiB). The maximum recommended task size is 1000 KiB.


Top specialized cast:
                     cast      genre  count
132103          Mel Blanc  Animation   1047
98140           Mel Blanc     Comedy    805
227339                Jr.      Drama    424
227017       Frank Welker  Animation    374
227712       Bess Flowers     Comedy    373
9205          Jack Mercer  Animation    360
86972   Jagathy Sreekumar      Drama    359
227018       Frank Welker     Family    335
227713       Bess Flowers      Drama    319
149048           Sukumari      Drama    312

Time: 7.59s


In [16]:
# ============================================================
# MAPREDUCE - Q5: DIRECTOR-CAST COMBINATIONS
# ============================================================

start_time = time.time()

records = movies[["director", "cast"]].to_dict("records")
rdd = sc.parallelize(records)

def emit_pairs(row):
    return [((d, c), 1) for d in row["director"] for c in row["cast"]]

result_q5 = (
    rdd.flatMap(emit_pairs)
       .reduceByKey(lambda a, b: a + b)
       .filter(lambda x: x[1] > 2)
       .map(lambda x: (x[0][0], x[0][1], x[1]))
)

df_q5_mr = pd.DataFrame(result_q5.collect(), columns=["director", "cast", "collabs"]) \
            .sort_values("collabs", ascending=False)

end_time = time.time()

print("Top collaborations:")
print(df_q5_mr.head(10))
print(f"\nTime: {end_time - start_time:.2f}s")

26/03/19 11:51:48 WARN TaskSetManager: Stage 4 contains a task of very large size (21289 KiB). The maximum recommended task size is 1000 KiB.


Top collaborations:
                   director                  cast  collabs
40865   Gilbert M. Anderson   Gilbert M. Anderson      320
20728          Friz Freleng             Mel Blanc      254
1874            Chuck Jones             Mel Blanc      235
129579           Kevin Dunn             John Cena      191
93103       Robert McKimson             Mel Blanc      179
165960           Kevin Dunn           Randy Orton      174
166356           Kevin Dunn          Glenn Jacobs      167
129577           Kevin Dunn          Mark Calaway      165
131031           Kevin Dunn            Paul Wight      155
165132           Kevin Dunn  Michael Hickenbottom      154

Time: 5.64s
